# CSE 151B Competition — Final Inference Notebook

This notebook is the grader-facing entry point. Running all cells loads the final vLLM pipeline and writes the required Kaggle CSV: `submission.csv`. The same logic is exposed as `run_inference()` in `run_inference.py`.

## 1. Environment Setup

Run this install cell only in a fresh environment. Leave `RUN_INSTALL = False` if dependencies are already installed.

In [ ]:
RUN_INSTALL = False

if RUN_INSTALL:
    import sys
    !{sys.executable} -m pip install -r requirements.txt

print("Environment setup cell complete.")

## 2. Imports & Configuration

All final hyperparameters are set inside `run_inference.py`; the notebook only chooses paths and calls the single entry point.

In [ ]:
import csv
import json
from pathlib import Path

from run_inference import MODEL_ID, run_inference

DATA_PATH = "data/private.jsonl"
OUTPUT_CSV = "submission.csv"
RESULTS_DIR = "results/private_run"

FRQ_MAX_TOKENS = 4096
FRQ_BATCH_SIZE = 5
MCQ_MAX_TOKENS = 32768

print(f"Model: {MODEL_ID}")
print(f"Data path: {DATA_PATH}")
print(f"Output CSV: {OUTPUT_CSV}")

## 3. Load the Dataset

This verifies that `private.jsonl` is present and shows the question mix before the long run starts.

In [ ]:
data_path = Path(DATA_PATH)
if not data_path.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place private.jsonl at data/private.jsonl before running inference."
    )

with data_path.open() as f:
    data = [json.loads(line) for line in f if line.strip()]

n_mcq = sum(bool(item.get("options")) for item in data)
n_frq = len(data) - n_mcq
print(f"Loaded {len(data)} questions ({n_mcq} MCQ, {n_frq} free-response).")
print("First id:", data[0].get("id") if data else None)

## 4. Run Full Inference

This is the required single entry point. It runs MCQ inference, FRQ inference, FRQ repair/post-processing, and writes `submission.csv`.

In [ ]:
submission_path = run_inference(
    data_path=DATA_PATH,
    output_csv=OUTPUT_CSV,
    results_dir=RESULTS_DIR,
    frq_max_tokens=FRQ_MAX_TOKENS,
    frq_batch_size=FRQ_BATCH_SIZE,
    mcq_max_tokens=MCQ_MAX_TOKENS,
)

print(f"Submission written to: {submission_path}")

## 5. Validate the CSV

The submission must contain exactly two columns: `id,response`, one row per private-set problem, with nonempty responses.

In [ ]:
csv_path = Path(OUTPUT_CSV)
if not csv_path.exists():
    raise FileNotFoundError(f"Missing output CSV: {csv_path}")

with csv_path.open(newline="") as f:
    rows = list(csv.DictReader(f))

ids = [row["id"] for row in rows]
blank = sum(not row.get("response", "").strip() for row in rows)
boxed = sum("\\boxed{" in row.get("response", "") for row in rows)

print(f"CSV rows: {len(rows)}")
print(f"Unique ids: {len(set(ids))}")
print(f"Blank responses: {blank}")
print(f"Rows containing boxed answers: {boxed}")

assert len(rows) == len(data), "CSV row count does not match dataset row count"
assert len(set(ids)) == len(rows), "CSV contains duplicate ids"
assert blank == 0, "CSV contains blank responses"
print("CSV validation passed.")